# function.py

In [31]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as mp
import scipy as sp
import cmath
import pandas as pd
#import seaborn as sb

In [32]:
def cphmat(vec,hoz,dim,N):
    """
    hmat Hankel matrix from a vector "vec"

    """
    try:
        vert = 1 + N[0] - hoz
    except:
        vert = 1 + N - hoz
    col = []
    for i in range(vert):
        row = []
        for j in range(hoz):
            row.append(vec[i + j])
        col.append(row)
    return cp.bmat(col)


def hankel_imputaion(data, lag, e, mask, dim=1, double=True, predata=None,srow=[0,0],**kawgs):
    """
    by a convex optimation of minimisation of the norm of hankle matrix of imputed variables
    with constraints = norm of mask of data and imputed variables less than a value e
    """
    if dim <= 1:
        print("monovar filling")
        N = len(data)
    else:
        print("Multivar filling")
        N = data.shape
    Yapp = cp.Variable(N)
    try:
        if dim <= 1:
            Yapp.value = predata.transpose()[0]
        else:
            Yapp.value = predata[:N[0]]
    except Exception as essa:
        print("no pre-data because of",essa,"\n")
    adjutvar = cphmat(Yapp,lag,dim,N)
    objective = cp.Minimize(cp.normNuc(adjutvar))
    if len(e) == 1:
        constraints = [cp.norm((data[mask] - Yapp[mask])) <= e[0]]
    else:
        constraints = [cp.norm((data[mask] - Yapp[mask])) <= e[0],contraint2(N,Yapp,e[1],srow)]   
    prob = cp.Problem(objective, constraints)
    prob.solve(**kawgs)
    return Yapp.value

def contraint2(N,reg,epsilon2,srow):
    fullmat = cp.matmul(formatix(N,True),reg)
    contrval = cp.norm(fullmat[N-srow[0]:N-srow[1]])
    return (contrval <= epsilon2)

def formatix(n,pure=False):
    mat = []
    for c in range(n):
        row = []
        for r in range(n):
            val = mkw(n)**(c*r)
            if pure:
                row.append(val)
            else:
                row.append(np.ma.round(val))
        mat.append(row)
    return cp.bmat(mat)

def mkw(bign):
    z = -1j
    trm = z*2*cmath.pi
    w = cmath.exp(trm/bign)
    return w

def construct(data, lag, dim):
    """
    sends a ValueError if lag is too big
    """
    N = data.shape
    Yapp = cp.Variable(N)
    #cp.normNuc(cphanker(Yapp[:lag], Yapp[lag:], dim)
    cp.normNuc(cphmat(Yapp, lag, dim, N))

In [33]:
def processing(data, batch, e, **kawgs):
    """
    calulates the mask, the lag, the shape
    """
    print("epsilon:",e)
    numpy_data = data.to_numpy()
    mask = data.notna()
    predata = data.copy().interpolate().to_numpy()
    try:
        N, dim = data.shape
    except ValueError:
         N = len(data)
         dim = 0
    print("your data has",N,"timesteps and",dim,"varibles")
    if batch == 0:
        lag = testmax_Lag(numpy_data,N,dim)
        filled = filling(numpy_data, mask, lag, e, dim, predata=predata, **kawgs)
    else:
        filled = batchfilling(numpy_data, mask, e, dim, batch, N, predata=predata, **kawgs)
    return filled

def filling(numpy_data, mask, lag, e, dim, **kawgs):
    """
    fill data using hankel imputaion method
    """
    if dim > 1:
        filled = hankel_imputaion(numpy_data, lag, e, mask, dim, **kawgs)
    elif dim == 1:
        filled = hankel_imputaion(
            numpy_data.transpose()[0], lag, e, mask.to_numpy().transpose()[0], **kawgs
        )
    else:
        filled = hankel_imputaion(
            numpy_data, lag, e, mask, **kawgs
        )
    return pd.DataFrame(filled)

def testmax_Lag(numpy_data,N,dim):
    """
    Test the Maximum number that lag can have a large lag gives a better results but too big have this error occurs:
    ValueError: All the input dimensions except for axis 0 must match exactly. cp.bmat(hank)
    """
    pnt = int(N / 2.8)
    for v in range(pnt):
        lag = pnt - v
        try:
            if dim > 1:
                construct(numpy_data, lag, dim)
            elif dim == 1:
                construct(numpy_data.transpose()[0], lag, dim)
        except ValueError as e:
            if lag % 5 == 0:
                print("less than", lag)
            if lag < int(N / 10):
                print(e)
                return
        else:
            print(lag)
            return lag

In [34]:
import random
def mkhle(data,percent):
    x, y = data.shape
    count = 0
    aim = round(x * y * percent)
    while count < aim:
        row = random.randint(1, x - 2)
        data.iloc[row, :] = np.nan
        count = data.isna().sum().sum()
        if count % 10 == 0:
            print(count, "(", aim, ")/", x * y, sep="")
    return data

def mk1dhls(data,percent):
    x = len(data)
    count = 0
    aim = round(x * percent)
    while count < aim:
        row = random.randint(1, x - 2)
        data.iloc[row] = np.nan
        count = data.isna().sum()
        if count % 10 == 0:
            print(count, "(", aim, ")/", x, sep="")
    return data

def fcshle(data,percent,focus):
    x = len(data)
    count = 0
    aim = round(x * percent)
    while count < aim:
        row = random.randint(focus, x - focus)
        data.iloc[row] = np.nan
        count = data.isna().sum()
        if count % 10 == 0:
            print(count, "(", aim, ")/", x, sep="")
    return data

In [ ]:
#data = np.array([1.3,1.1,1,1.2,1.3,1.4,1.54,1.9,1.81,1.43,1.61,2.1,2.3,1.7,1.5,1.4,1.2,1.3])
data = pd.read_csv("Arsamples.csv")
data.columns = data.columns.astype("float")

In [2]:
from statsmodels.tsa.seasonal import seasonal_decompose
from math import sqrt
from sklearn.metrics import mean_squared_error

In [3]:
def acana(org,fill,perod):
    #mod = 'multiplicative'
    maxis = len(org) - perod
    mod = "additve"
    decomp = seasonal_decompose(org,period=perod, model=mod)
    defill = seasonal_decompose(fill,period=perod, model=mod)
    noisecomp = decomp.seasonal+decomp.resid
    noisefill = defill.seasonal+defill.resid
    #return defill.trend
    trend_score = mean_squared_error(defill.trend[perod:maxis],decomp.trend[perod:maxis])
    noise_score = abs(noisefill[perod:maxis].std() - noisecomp[perod:maxis].std())
    print(trend_score, noise_score)
    return trend_score, noise_score

In [4]:
def testfourer(fulldat,eep,alpha,per,verbs=True,rot=7):
    hodf = mk1dhls(fulldat[alpha].copy(),per)
    lowres = processing(hodf, 0, eep,verbose=verbs)
    nofores = processing(hodf, 0, [eep[0]],verbose=verbs)
    arset = pd.concat([lowres,
                  nofores,
                  fulldat[alpha],
                  hodf,
                  ],axis=1)
    arset.columns =["forier","normal","org","remove"]
    fr = acana(arset["org"][1:53],arset["forier"][1:53],rot)
    nr = acana(arset["org"][1:53],arset["normal"][1:53],rot)
    return arset,fr,nr
    
def plotres(dats):
    pt,ax = mp.subplots(4)
    for i in dats.columns:
        ax[i].plot(dats[i][20:])

In [ ]:
def dtbloc(fchodf,lenght,div=1):
    dt = []
    starttim = time.time()
    for i in lenght:
        res = processing(fchodf, 0, [0.0001,i/div],srow=[1,0])
        print(time.time() - starttim,i,lenght[-1])
        starttim = time.time()
        dt.append(res)
    togther = pd.concat(dt,axis=1)
    togther.columns = lenght
    return togther

In [ ]:
def getana(data,org,sea=7):
    analist = []
    for i in data.columns:
        trend_score, noise_score = acana(org,data[i],sea)
        analist.append([msecal(org,data[i]),trend_score, noise_score])
    return pd.DataFrame(analist,index=data.columns,columns=["trend score", "noise score","mse"])